# Waterinfo data acquisition demo

Scratchpad for the demo script based on the [`rws-waterinfo`](https://pypi.org/project/rws-waterinfo/) package.

Request format expected by `rw.get_data` (list of lists, 7 items each):

`[compartiment_code, eenheid_code, meetapparaat_code, grootheid_code, locatie_code, start_date, end_date]`

In [48]:
import pandas as pd
import rws_waterinfo as rw

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. Fetch the catalog

Takes roughly 10 seconds and returns ~211k records.

In [49]:
catalog = rw.get_catalog()
# check the initial catalog content
print(catalog.shape)
catalog.head()

(211338, 40)


,AquoMetaData_MessageID,Locatie_MessageID,Code,Coordinatenstelsel,Lat,Lon,Naam,Omschrijving,Parameter_Wat_Omschrijving,ProcesType,BemonsteringsApparaat.Code,BemonsteringsApparaat.Omschrijving,BemonsteringsMethode.Code,BemonsteringsMethode.Omschrijving,BemonsteringsSoort.Code,BemonsteringsSoort.Omschrijving,BioTaxon.Code,BioTaxon.Omschrijving,Compartiment.Code,Compartiment.Omschrijving,Eenheid.Code,Eenheid.Omschrijving,Grootheid.Code,Grootheid.Omschrijving,Hoedanigheid.Code,Hoedanigheid.Omschrijving,MeetApparaat.Code,MeetApparaat.Omschrijving,Orgaan.Code,Orgaan.Omschrijving,Parameter.Code,Parameter.Omschrijving,Typering.Code,Typering.Omschrijving,WaardeBepalingsMethode.Code,WaardeBepalingsMethode.Omschrijving,WaardeBepalingsTechniek.Code,WaardeBepalingsTechniek.Omschrijving,WaardeBewerkingsMethode.Code,WaardeBewerkingsMethode.Omschrijving
0,1,129,kornwerderzand.waddenzee.buitenhaven,ETRS89,53.074590,5.334757,Kornwerderzand Waddenzee buitenhaven,Kornwerderzand Waddenzee buitenhaven,Percentielen per etmaal Levendigheid 50 percen...,meting,8000,NVT,NVT,NVT,SB,Steekbemonstering,NVT,NVT,OW,Oppervlaktewater,cm2,vierkante centimeter,50%_L,50 percentiel van de levendigheid,NVT,NVT,10042,other:Vlotterniveaumeter - type DNM,NVT,NVT,NVT,NVT,LEVDHD,Levendigheid,other:F026,Gem. levendigheid mbv Chebyshev filter vorige ...,NVT,NVT,other:%24H,Percentielen per etmaal
1,1,144,zwolle.ijssel,ETRS89,52.508600,6.053000,"Zwolle, IJssel",voorheen Katerveer,Percentielen per etmaal Levendigheid 50 percen...,meting,8000,NVT,NVT,NVT,SB,Steekbemonstering,NVT,NVT,OW,Oppervlaktewater,cm2,vierkante centimeter,50%_L,50 percentiel van de levendigheid,NVT,NVT,10042,other:Vlotterniveaumeter - type DNM,NVT,NVT,NVT,NVT,LEVDHD,Levendigheid,other:F026,Gem. levendigheid mbv Chebyshev filter vorige ...,NVT,NVT,other:%24H,Percentielen per etmaal
2,1,683,lelystad.houtribsluis.zuid,ETRS89,52.526351,5.433916,"Lelystad, Houtribsluis, zuid","Lelystad, Houtribsluis, zuid",Percentielen per etmaal Levendigheid 50 percen...,meting,8000,NVT,NVT,NVT,SB,Steekbemonstering,NVT,NVT,OW,Oppervlaktewater,cm2,vierkante centimeter,50%_L,50 percentiel van de levendigheid,NVT,NVT,10042,other:Vlotterniveaumeter - type DNM,NVT,NVT,NVT,NVT,LEVDHD,Levendigheid,other:F026,Gem. levendigheid mbv Chebyshev filter vorige ...,NVT,NVT,other:%24H,Percentielen per etmaal
3,1,738,zutphen.ijssel,ETRS89,52.154000,6.182000,"Zutphen, IJssel",noord,Percentielen per etmaal Levendigheid 50 percen...,meting,8000,NVT,NVT,NVT,SB,Steekbemonstering,NVT,NVT,OW,Oppervlaktewater,cm2,vierkante centimeter,50%_L,50 percentiel van de levendigheid,NVT,NVT,10042,other:Vlotterniveaumeter - type DNM,NVT,NVT,NVT,NVT,LEVDHD,Levendigheid,other:F026,Gem. levendigheid mbv Chebyshev filter vorige ...,NVT,NVT,other:%24H,Percentielen per etmaal
4,1,837,culemborg,ETRS89,51.961000,5.214000,Culemborg,brug,Percentielen per etmaal Levendigheid 50 percen...,meting,8000,NVT,NVT,NVT,SB,Steekbemonstering,NVT,NVT,OW,Oppervlaktewater,cm2,vierkante centimeter,50%_L,50 percentiel van de levendigheid,NVT,NVT,10042,other:Vlotterniveaumeter - type DNM,NVT,NVT,NVT,NVT,LEVDHD,Levendigheid,other:F026,Gem. levendigheid mbv Chebyshev filter vorige ...,NVT,NVT,other:%24H,Percentielen per etmaal


In [50]:
# have a look at what is available to filter on
# sorted(catalog.columns)

## 2. Build the location and parameter subsets

In [51]:
method_code = "other:X156"

# create unique list of locations based on the 'WaardeBepalingsMethode.Code' column
# that matches 'other:X156'
loccatalog = (
    catalog[catalog["WaardeBepalingsMethode.Code"] == method_code]
    .groupby("Code")
    .first()
    .reset_index()
)

# create unique list of parameters based on the 'WaardeBepalingsMethode.Code' column
# that matches 'other:X156'
paramcatalog = (
    catalog[catalog["WaardeBepalingsMethode.Code"] == method_code]
    .groupby("Parameter_Wat_Omschrijving")
    .first()
    .reset_index()
)

print(f"locations: {len(loccatalog)}, parameters: {len(paramcatalog)}")

locations: 4, parameters: 171


## 3. Single request

Start with one location to check the response before looping.

In [30]:
startdate = "2022-12-31"
enddate = "2023-12-31"
grootheid_code = "AANTPLTE"
compartiment_code = "OR"

loc = loccatalog["Code"].iloc[0]

params = [[compartiment_code, "", "", grootheid_code, loc, startdate, enddate]]
data = rw.get_data(params=params, return_df=True, max_workers=8)

print(loc, data.shape)

Downloaded 1/1
bergenaanzee.standafvalmeetnet (584, 51)


## 4. Loop over all locations

> **Note on the original demo script.** It looped over locations *and* parameters, but the
> request was built only from `grootheid_code` and `loc` — the `param` variable was never
> used. That fetched the same data 171 times per location. The loop below iterates over
> locations only, which is what `downloader.py` does as well.

In [52]:
frames = []

for loc in loccatalog["Code"]:
    print(f"Processing location: {loc}")
    params = [[compartiment_code, "", "", grootheid_code, loc, startdate, enddate]]
    try:
        data = rw.get_data(params=params, return_df=True, max_workers=8)
    except Exception as exc:
        print(f"  failed: {exc}")
        continue
    if data is None or len(data) == 0:
        print("  no data returned")
        continue
    data = data.copy()
    data["krm_location_code"] = loc
    frames.append(data)
    print(f"  {len(data)} rows")

combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
combined.shape

Processing location: bergenaanzee.standafvalmeetnet
Downloaded 1/1
  584 rows
Processing location: noordwijk.strandafvalmeetnet
Downloaded 1/1
  584 rows
Processing location: terschelling.strandafvalmeetnet
Downloaded 1/1
  584 rows
Processing location: veere.strandafvalmeetnet
Downloaded 1/1
  584 rows


(2336, 52)

## 5. Write results to the data folder

The repository root is mounted at `/work`, so this writes to `data/waterinfo/notebook/` on the host.

In [53]:
from pathlib import Path

out_dir = Path("/work/data/waterinfo/notebook")
out_dir.mkdir(parents=True, exist_ok=True)

target = out_dir / f"{grootheid_code}_combined.csv"
combined.to_csv(target, index=False)
print(f"wrote {len(combined)} rows to {target}")

wrote 2336 rows to /app/data/waterinfo/notebook/AANTPLTE_combined.csv


## 6. Compare with the script settings

`download.py` reads the same values from `config.toml`, so you can check the notebook against it.

In [54]:
import tomllib

with open("../config.toml", "rb") as f:
    cfg = tomllib.load(f)

cfg

DownloadConfig(start_date='2022-12-31', end_date='2023-12-31', method_code='other:X156', grootheid_code='AANTPLTE', compartiment_code='OR', eenheid_code='', meetapparaat_code='', output_dir=PosixPath('/app/data'), max_workers=8, limit_locations=1, overwrite=False, save_full_catalog=False, log_level='INFO')
